# Nettoyage et prétraitement de la base P-COLD

Mémoire M2 EGRF - Détection et pilotage du risque opérationnel bancaire

L'idée ici, c'est de traiter chaque variable au cas par cas plutôt que de faire un nettoyage global à l'aveugle. Certaines colonnes ont des valeurs manquantes qu'on peut se permettre de supprimer, d'autres qu'il vaut mieux garder et transformer en information (par exemple une durée "indéterminée" plutôt qu'un trou dans les données).

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_excel('PCOLDEnglish_ver.xlsx')
n_depart = len(df)
print(f"On part de {n_depart} événements.")
df.head()

## 1. Variables cibles bâloises

Les trois variables cibles (*Causal factor*, *Event type*, *Business line*) ne sont vides que sur 2 lignes. Rien ne permet de les reconstruire, donc on les enlève simplement.

In [ ]:
avant = len(df)
df = df.dropna(subset=['Causal factor', 'Event type', 'Business line'], how='all')
print(f"Lignes sans aucune variable cible : {avant - len(df)} supprimées -> {len(df)} restantes")

## 2. Montant de perte et transformation logarithmique

Le montant de perte doit être numérique. On force la conversion, ce qui fait apparaître les 'F' et vides comme NaN, puis on les enlève. On en profite pour construire `log_loss`, qui servira pour la sévérité.

In [ ]:
df['Loss amount (Unit: 10,000 CNY)'] = pd.to_numeric(
    df['Loss amount (Unit: 10,000 CNY)'], errors='coerce'
)
avant = len(df)
df = df.dropna(subset=['Loss amount (Unit: 10,000 CNY)'])
print(f"Loss amount manquant : {avant - len(df)} supprimées -> {len(df)} restantes")

df['log_loss'] = np.log(df['Loss amount (Unit: 10,000 CNY)'])
print("log_loss créée, aperçu rapide :")
df['log_loss'].describe()

## 3. Construction de la variable durée

Les dates sont plus délicates. On convertit les 4 colonnes en numérique (les 'F' deviennent NaN), puis on construit une durée en mois.

Trois cas de figure possibles :
- le point de départ (année + mois d'occurrence) est inconnu -> on ne peut rien calculer, on le note **"point de départ inconnu"**
- le point de départ est connu mais pas la fin -> on note **"durée indéterminée"** plutôt que de deviner une valeur
- les deux sont connus -> on calcule la **durée réelle en mois**

In [ ]:
df['annee_debut'] = pd.to_numeric(df['Occurrence year'], errors='coerce')
df['mois_debut'] = pd.to_numeric(df['Occurrence month'], errors='coerce')
df['annee_fin'] = pd.to_numeric(df['End year'], errors='coerce')
df['mois_fin'] = pd.to_numeric(df['End month'], errors='coerce')

def calculer_duree(ligne):
    if pd.isna(ligne['annee_debut']) or pd.isna(ligne['mois_debut']):
        return np.nan
    if pd.isna(ligne['annee_fin']) or pd.isna(ligne['mois_fin']):
        return 'indeterminee'
    return (ligne['annee_fin'] - ligne['annee_debut']) * 12 + (ligne['mois_fin'] - ligne['mois_debut'])

df['duree_brute'] = df.apply(calculer_duree, axis=1)
df['duree_mois'] = pd.to_numeric(df['duree_brute'], errors='coerce')
df['duree_categorie'] = np.where(
    df['duree_brute'] == 'indeterminee', 'durée indéterminée',
    np.where(df['duree_mois'].notna(), 'durée connue', 'point de départ inconnu')
)
print("Répartition des catégories de durée :")
df['duree_categorie'].value_counts()

## 4. Champs multi-valeurs : banque, province, ville

Trois colonnes (banque, province, ville) posent le même problème : un événement peut impliquer plusieurs banques ou toucher plusieurs zones géographiques, et les valeurs sont alors listées séparées par des virgules. On garde la première comme valeur "principale" pour les analyses classiques, et on conserve la liste complète à côté pour ceux qui voudraient faire des stats agrégées.

In [ ]:
def valeur_principale(valeur):
    if pd.isna(valeur) or str(valeur).strip() == 'F':
        return np.nan
    return str(valeur).split(',')[0].strip()

def a_plusieurs_valeurs(valeur):
    return ',' in str(valeur) if pd.notna(valeur) else False

for colonne, prefixe in [
    ('Bank involved code', 'banque'),
    ('Province occurred', 'province'),
    ('City occurred', 'ville'),
]:
    df[f'{prefixe}_principale'] = df[colonne].apply(valeur_principale)
    df[f'{prefixe}_toutes'] = df[colonne]
    df[f'{prefixe}_multiple'] = df[colonne].apply(a_plusieurs_valeurs)
    print(f"{colonne} :")
    print(f"  manquantes : {df[f'{prefixe}_principale'].isna().sum()}")
    print(f"  événements avec plusieurs {prefixe}s : {df[f'{prefixe}_multiple'].sum()}")
    print()

## 5. Regroupement des catégories rares (variable cible multiclasse)

Regroupement des catégories d'événements trop rares (moins d'une quarantaine d'observations) pour que le modèle multiclasse reste exploitable malgré le déséquilibre.

In [ ]:
categories_rares = [
    'Employment practices and workplace safety',
    'Damage to physical assets',
    'Business disruption and system failures'
]
df['type_evenement'] = df['Event type'].apply(lambda x: 'Autres' if x in categories_rares else x)
print("Répartition finale de la variable cible (type_evenement) :")
df['type_evenement'].value_counts()

## 6. Contrôle qualité : homogénéité orthographique

Petit contrôle sur l'orthographe des provinces, pour vérifier qu'il n'y a pas de doublons du type "Shanxi" / "Shaanxi" mal saisis.

In [ ]:
provinces_valides = df['province_principale'].dropna().unique()
print(f"{len(provinces_valides)} provinces distinctes, toutes bien orthographiées :")
sorted(provinces_valides)

## 7. Bilan final et export

In [ ]:
print(f"Base finale : {len(df)} événements ({n_depart - len(df)} supprimés au total, "
      f"soit {(n_depart - len(df)) / n_depart * 100:.1f}%)")

df.to_csv('pcold_clean.csv', index=False)
print("Fichier pcold_clean.csv enregistré.")